# Chess — First-Visit Monte Carlo **Control** (ε-greedy)

**Corrected policy-learning version of the First-Visit Monte Carlo Chess lab.** Both White and Black now learn to choose moves, rather than always making random moves.

| RL component | Implementation |
|---|---|
| Environment | Legal 8×8 chess (`python-chess`) |
| State `s` | Abstract chess position: phase, material balance, check status, queen presence |
| Action `a` | Category of a legal move (e.g. pawn advance, capture, check, castle) |
| Policy `π` | **ε-greedy**: explore legal moves with probability ε; otherwise select highest estimated `Q(s,a)` |
| Reward | +1 for a win, 0 for a draw, −1 for a loss (each player's own perspective) |
| Algorithm | **First-Visit Monte Carlo Control**, updating action-values from completed episode returns |

**Course link:** Chapter 5, pp. 39–42 (First-Visit Monte Carlo and Monte Carlo Control).

**Honest limitations:** Chess is too large for a full board-position × exact-move tabular Q-table. This lab groups positions and moves into interpretable categories, so the policy can generalize across games. It is an **educational learning agent, not a competitive chess engine**. For manageable Colab runtime, it also caps each game at a fixed number of half-moves and explicitly adjudicates unfinished games by material; those outcomes are *not* official chess results.

## 1. Install libraries and configure training

Run all cells in sequence. No Pygame desktop window is required.

In [ ]:
%pip -q install python-chess

In [ ]:
import random
from collections import Counter, defaultdict

import chess
import chess.svg
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, SVG
from tqdm.auto import trange

SEED = 42
TRAIN_EPISODES = 800      # For more learning, try 2_000 or 5_000
MAX_PLIES = 120          # Finite-horizon training task: 120 half-moves per episode
GAMMA = 1.0             # Full episodic return, as in First-Visit MC prediction
EPS_START = 1.0         # Initially explore every time
EPS_END = 0.05          # Maintain a small amount of exploration
EVAL_GAMES = 30         # Each learned policy vs a random legal-move opponent
MATERIAL_WIN_MARGIN = 2.0  # Material advantage in pawn units for cap adjudication

rng = random.Random(SEED)
print("Method: First-Visit MC CONTROL | policy: epsilon-greedy")
print(f"Training: {TRAIN_EPISODES} games, maximum {MAX_PLIES} plies each")

## 2. Define the chess state and move categories

**Why abstract the board?** A full chess FEN is usually encountered only once. A Q-table indexed by exact FEN and exact move cannot learn much from a few hundred games. We therefore teach the agents with **state and action categories** shared across many chess positions.

A category is a **feature description**, not a hardcoded priority. For example, the code does not tell the agent that capturing a queen is automatically better: it must learn an estimated return for choosing that category from past episodes.

In [ ]:
PIECE_VALUES = {
    chess.PAWN: 1.0, chess.KNIGHT: 3.0, chess.BISHOP: 3.0,
    chess.ROOK: 5.0, chess.QUEEN: 9.0, chess.KING: 0.0
}
CENTER = {chess.D4, chess.E4, chess.D5, chess.E5}


def material(board, color):
    return sum(PIECE_VALUES[t] * len(board.pieces(t, color)) for t in PIECE_VALUES)


def state_key(board):
    """A small, reusable state representation from the current player's view."""
    me = board.turn
    diff = material(board, me) - material(board, not me)
    bucket = max(-3, min(3, round(diff / 3)))
    phase = min(2, board.ply() // 30)
    my_queen = bool(board.pieces(chess.QUEEN, me))
    their_queen = bool(board.pieces(chess.QUEEN, not me))
    return (phase, bucket, board.is_check(), my_queen, their_queen)


def action_category(board, move):
    """Group legal chess actions so Monte Carlo can learn general preferences."""
    mover = board.piece_at(move.from_square)
    if move.promotion:
        return "promotion"
    if board.is_castling(move):
        return "castle"
    if board.is_capture(move):
        target = board.piece_at(move.to_square)  # None for en passant
        kind = chess.PAWN if target is None else target.piece_type
        if kind == chess.QUEEN:
            return "capture_queen"
        if kind == chess.ROOK:
            return "capture_rook"
        if kind in (chess.BISHOP, chess.KNIGHT):
            return "capture_minor"
        return "capture_pawn"
    if board.gives_check(move):
        return "give_check"
    if mover.piece_type == chess.PAWN:
        return "center_pawn" if move.to_square in CENTER else "pawn_push"
    if mover.piece_type in (chess.BISHOP, chess.KNIGHT):
        return "minor_move"
    if mover.piece_type == chess.ROOK:
        return "rook_move"
    if mover.piece_type == chess.QUEEN:
        return "queen_move"
    return "king_move"


# Verify the features on the initial position.
start_board = chess.Board()
assert len(state_key(start_board)) == 5
assert all(action_category(start_board, m) for m in start_board.legal_moves)
print("Initial state:", state_key(start_board))
print("Available action categories:", sorted({action_category(start_board, m) for m in start_board.legal_moves}))

## 3. Define the ε-greedy policy for White and Black

For legal moves `a` in state `s`:

\[
\pi_\epsilon(a|s) = \begin{cases}
\text{random legal move} & \text{with probability } \epsilon,\\
\text{move maximizing } Q(s,a) & \text{otherwise.}
\end{cases}
\]

Q-values are **empirical means of First-Visit episode returns**. The two players have independent Q-tables.

In [ ]:
class MCAgent:
    def __init__(self, name):
        self.name = name
        self.returns_sum = defaultdict(float)
        self.count = defaultdict(int)

    def q(self, state, action_group):
        key = (state, action_group)
        n = self.count.get(key, 0)
        return self.returns_sum[key] / n if n else 0.0

    def choose(self, board, epsilon, generator):
        state = state_key(board)
        legal = list(board.legal_moves)
        if not legal:
            raise ValueError("No legal action in a terminal state")
        options = [(m, action_category(board, m)) for m in legal]
        scores = [self.q(state, group) for _, group in options]

        if generator.random() < epsilon:
            index = generator.randrange(len(options))
            mode = "EXPLORE (random legal move)"
        else:
            high = max(scores)
            best = [i for i, score in enumerate(scores) if abs(score - high) < 1e-12]
            index = generator.choice(best)
            mode = "EXPLOIT (highest learned Q)"

        move, group = options[index]
        return {
            "move": move, "state": state, "category": group,
            "mode": mode, "epsilon": epsilon, "q_before": scores[index],
            "visits_before": self.count.get((state, group), 0),
            "options": {g: self.q(state, g) for _, g in options},
        }

    def learn_first_visit(self, trajectory, reward, terminal_ply=None, gamma=GAMMA):
        """First occurrence of each (abstract state, move category) per episode."""
        visited = set()
        updates = 0
        total_plies = terminal_ply if terminal_ply is not None else max(
            (row["ply"] for row in trajectory), default=-1
        ) + 1
        for row in trajectory:
            key = (row["state"], row["category"])
            if key in visited:
                continue
            visited.add(key)
            # Reward happens at the end of the complete episode.
            G = reward * gamma ** (total_plies - row["ply"] - 1)
            self.returns_sum[key] += G
            self.count[key] += 1
            updates += 1
        return updates


white_agent = MCAgent("White")
black_agent = MCAgent("Black")

# First-visit test: repeated state-action in one episode gets counted once.
toy = MCAgent("test")
key_state = (0, 0, False, True, True)
toy.learn_first_visit([
    {"state": key_state, "category": "pawn_push", "ply": 0},
    {"state": key_state, "category": "pawn_push", "ply": 2},
], reward=1.0, gamma=1.0)
assert toy.count[(key_state, "pawn_push")] == 1
assert toy.q(key_state, "pawn_push") == 1.0
print("First-Visit MC control unit test passed.")

## 4. Environment, rewards, and complete episodes

Official game results are respected when available. At **120 plies**, if no chess result has occurred, the game is explicitly ended using a material comparison for this teaching environment (winning margin: `MATERIAL_WIN_MARGIN` pawns). **These cap-adjudicated results are not official chess wins**. This ensures the Monte Carlo algorithm receives a return for every finite episode.

In [ ]:
def episode_result(board):
    outcome = board.outcome(claim_draw=True)
    if outcome is not None:
        if outcome.winner is None:
            return 0, "official_draw"
        return (+1 if outcome.winner == chess.WHITE else -1), "official_win"

    # Educational horizon-based adjudication, NOT standard chess result.
    difference = material(board, chess.WHITE) - material(board, chess.BLACK)
    if difference >= MATERIAL_WIN_MARGIN:
        return +1, "material_cap"
    if difference <= -MATERIAL_WIN_MARGIN:
        return -1, "material_cap"
    return 0, "material_cap"


def play_episode(agent_white, agent_black, epsilon, generator):
    board = chess.Board()
    trace = []
    for ply in range(MAX_PLIES):
        if board.is_game_over(claim_draw=True):
            break
        color = board.turn
        agent = agent_white if color == chess.WHITE else agent_black
        if agent is None:  # Random baseline for evaluation ONLY.
            move = generator.choice(list(board.legal_moves))
            choice = {
                "move": move, "state": state_key(board),
                "category": action_category(board, move),
                "mode": "RANDOM BASELINE", "epsilon": 1.0,
                "q_before": None, "visits_before": 0, "options": {},
            }
        else:
            choice = agent.choose(board, epsilon, generator)
        choice.update({
            "color": color, "ply": ply, "san": board.san(choice["move"]),
        })
        trace.append(choice)
        board.push(choice["move"])

    white_reward, ending = episode_result(board)
    return {
        "trace": trace, "board": board, "white_reward": white_reward,
        "ending": ending, "plies": len(trace),
    }


def train_one_episode(white, black, epsilon, generator):
    game = play_episode(white, black, epsilon, generator)
    for color, agent in [(chess.WHITE, white), (chess.BLACK, black)]:
        my_trace = [r for r in game["trace"] if r["color"] == color]
        reward = game["white_reward"] * (1 if color == chess.WHITE else -1)
        # 'ply' is the global index; discount is applied to terminal reward.
        agent.learn_first_visit(my_trace, reward, terminal_ply=game["plies"], gamma=GAMMA)
    return game


smoke = play_episode(None, None, epsilon=1.0, generator=random.Random(100))
assert len(smoke["trace"]) > 0
assert smoke["white_reward"] in (-1, 0, 1)
assert all(row["move"] in chess.Board().legal_moves for row in smoke["trace"][:1])
print("Smoke test:", smoke["plies"], "plies |", smoke["ending"], "| reward", smoke["white_reward"])

## 5. Train both players — First-Visit MC Control

The exploration rate decreases gradually. After **every finished episode**, the algorithm evaluates the observed state–action returns and updates both policies.

In [ ]:
def training_epsilon(index, total):
    fraction = index / max(1, total - 1)
    return EPS_START * (EPS_END / EPS_START) ** fraction


def train(num_episodes=TRAIN_EPISODES):
    white_agent.returns_sum.clear(); white_agent.count.clear()
    black_agent.returns_sum.clear(); black_agent.count.clear()
    training_rng = random.Random(SEED)
    results = Counter()
    batches = []
    wins = []
    lengths = []

    for i in trange(num_episodes, desc="First-Visit MC Control training"):
        epsilon = training_epsilon(i, num_episodes)
        game = train_one_episode(white_agent, black_agent, epsilon, training_rng)
        label = "White win" if game["white_reward"] > 0 else ("Black win" if game["white_reward"] < 0 else "Draw")
        results[label] += 1
        results[game["ending"]] += 1
        wins.append(game["white_reward"])
        lengths.append(game["plies"])
        if (i + 1) % max(1, num_episodes // 8) == 0:
            batches.append((i + 1, epsilon, len(white_agent.count), len(black_agent.count)))

    print(f"\nTraining complete: {num_episodes:,} episodes")
    print(f"White wins={results['White win']}, Black wins={results['Black win']}, Draws={results['Draw']}")
    print(f"Official results={results['official_win'] + results['official_draw']}, "
          f"material-cap adjudications={results['material_cap']}")
    print(f"Distinct learned (state, action-category) pairs: "
          f"White={len(white_agent.count)}, Black={len(black_agent.count)}")
    print(f"Average episode length: {np.mean(lengths):.1f} plies")
    return results, wins, lengths, batches


training_results, training_outcomes, training_lengths, checkpoints = train()
print("\nCheckpoints (episode, epsilon, White Q entries, Black Q entries):")
for point in checkpoints:
    print(point)

## 6. Plot training and learned action preferences

The charts show training outcomes and the changing exploration rate. They do **not** alone prove strong play; the next section evaluates both agents against a random opponent.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(["White wins", "Black wins", "Draws"],
            [training_results["White win"], training_results["Black win"], training_results["Draw"]])
axes[0].set_title("Self-play episode outcomes")
axes[0].set_ylabel("Episodes")
axes[1].plot(range(TRAIN_EPISODES), [training_epsilon(i, TRAIN_EPISODES) for i in range(TRAIN_EPISODES)])
axes[1].set_title("Epsilon decay: exploration → exploitation")
axes[1].set_xlabel("Training episode")
axes[1].set_ylabel("Epsilon")
plt.tight_layout()
plt.show()


def learned_categories(agent, top=12):
    tally = defaultdict(lambda: [0.0, 0])
    for (state, group), count in agent.count.items():
        tally[group][0] += agent.returns_sum[(state, group)]
        tally[group][1] += count
    return sorted(((group, s / n, n) for group, (s, n) in tally.items() if n),
                  key=lambda x: x[1], reverse=True)[:top]

for agent in [white_agent, black_agent]:
    print(f"\n{agent.name} — learned average returns by move category (descriptive only):")
    for group, average, observations in learned_categories(agent):
        print(f"  {group:20s} avg return={average:+.3f}, visits={observations}")

## 7. Evaluate learned policy vs random baseline

During evaluation the trained agent uses **ε = 0** (greedy). The opponent is a random legal-move player used only as a benchmark. **Cap-adjudicated wins** are distinguished from official chess results.

In [ ]:
def evaluate(learner, agent_color, games=EVAL_GAMES, seed=2027):
    generator = random.Random(seed)
    scores = Counter()
    for _ in range(games):
        white = learner if agent_color == chess.WHITE else None
        black = learner if agent_color == chess.BLACK else None
        game = play_episode(white, black, epsilon=0.0, generator=generator)
        reward = game["white_reward"] if agent_color == chess.WHITE else -game["white_reward"]
        scores["Win" if reward == 1 else ("Loss" if reward == -1 else "Draw")] += 1
        scores[game["ending"]] += 1
    return scores


baseline_white = evaluate(MCAgent("Untrained White"), chess.WHITE)
baseline_black = evaluate(MCAgent("Untrained Black"), chess.BLACK)
white_eval = evaluate(white_agent, chess.WHITE)
black_eval = evaluate(black_agent, chess.BLACK)
print("White untrained vs RANDOM opponent:", dict(baseline_white))
print("White trained   vs RANDOM opponent:", dict(white_eval))
print("Black untrained vs RANDOM opponent:", dict(baseline_black))
print("Black trained   vs RANDOM opponent:", dict(black_eval))
print("\nThese are capped-horizon educational evaluations, not chess Elo ratings.")

## 8. Demonstrate a game — explain every AI move

**Both learned policies** now play with `epsilon=0` (always exploitation), without any hand-coded tactic priorities. Every move prints its action category, the learned Q-value and the number of observations behind it. Positions with zero visits are unknown and can yield ties.

In [ ]:
def show_learned_game(max_steps=12, size=310, seed=731):
    demo = play_episode(white_agent, black_agent, epsilon=0.0,
                        generator=random.Random(seed))
    board = chess.Board()
    print("Learnt-policy demonstration: White vs Black")
    print("Entire game:", demo["plies"], "plies |", demo["ending"],
          "| White reward:", demo["white_reward"])

    for idx, record in enumerate(demo["trace"][:max_steps], start=1):
        side = "White" if record["color"] == chess.WHITE else "Black"
        print(f"\nStep {idx}: {side} → {record['san']}")
        print(f"Policy: {record['mode']} | epsilon={record['epsilon']:.2f}")
        print(f"Chosen action category: {record['category']}")
        print(f"Q(state, category) BEFORE move = {record['q_before']:+.3f} "
              f"({record['visits_before']} previous first-visits)")
        top = sorted(record["options"].items(), key=lambda item: item[1], reverse=True)[:6]
        print("Legal action-category Q values:", ", ".join(f"{g}={v:+.2f}" for g, v in top))
        board.push(record["move"])
        display(SVG(chess.svg.board(board=board, lastmove=record["move"], size=size)))
    return demo


demo = show_learned_game(max_steps=10)

## 9. Optional: physical-style visualization in Colab

This keeps the original TP's idea of piece motion with **Pymunk**, but animates inside Colab as an inline GIF (no desktop Pygame window). The physics is **visual only**; legal chess moves and reinforcement learning are governed by `python-chess`.

In [ ]:
%pip -q install pymunk Pillow

In [ ]:
import io
from PIL import Image, ImageDraw, ImageFont
from IPython.display import Image as IPyImage
import pymunk

CELL = 60
BOARD_PIXELS = 8 * CELL

def center_for_square(sq):
    x = (chess.square_file(sq) + 0.5) * CELL
    y = (7.5 - chess.square_rank(sq)) * CELL
    return float(x), float(y)


def make_chess_frame(board, moving=None):
    """Render the board with one optional animated circular piece token."""
    img = Image.new("RGB", (BOARD_PIXELS, BOARD_PIXELS), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", 24)
    except OSError:
        font = ImageFont.load_default()

    for rank in range(8):
        for file in range(8):
            xy = [file * CELL, rank * CELL, (file + 1) * CELL, (rank + 1) * CELL]
            fill = "#E6D8C0" if (file + rank) % 2 == 0 else "#789195"
            draw.rectangle(xy, fill=fill)

    hide_square = moving["from_sq"] if moving else None
    for sq, piece in board.piece_map().items():
        if sq == hide_square:
            continue
        x, y = center_for_square(sq)
        fill = "#FCFCFA" if piece.color == chess.WHITE else "#29323A"
        ink = "#1F2937" if piece.color == chess.WHITE else "#F8FAFC"
        draw.ellipse([x - 21, y - 21, x + 21, y + 21],
                     fill=fill, outline="#111827", width=2)
        draw.text((x, y), piece.symbol().upper(), fill=ink, font=font,
                  anchor="mm")

    if moving:
        x, y = moving["xy"]
        piece = moving["piece"]
        fill = "#FCFCFA" if piece.color == chess.WHITE else "#29323A"
        ink = "#1F2937" if piece.color == chess.WHITE else "#F8FAFC"
        draw.ellipse([x - 21, y - 21, x + 21, y + 21],
                     fill=fill, outline="#FBBF24", width=3)
        draw.text((x, y), piece.symbol().upper(), fill=ink, font=font,
                  anchor="mm")
    return img


def show_physics_gif(episode, max_moves=4, frames_per_move=14):
    """Animate legal chess moves; Pymunk produces the moving piece's trajectory."""
    if episode is None:
        print("No completed game available to animate.")
        return

    board = chess.Board()
    frames = []
    for move in episode["moves"][:max_moves]:
        moving_piece = board.piece_at(move.from_square)
        from_xy = center_for_square(move.from_square)
        target_xy = center_for_square(move.to_square)

        space = pymunk.Space()
        space.gravity = (0, 0)
        space.damping = 0.7
        mass, radius = 1.0, 20
        body = pymunk.Body(mass, pymunk.moment_for_circle(mass, 0, radius))
        body.position = from_xy
        shape = pymunk.Circle(body, radius)
        space.add(body, shape)

        for _ in range(frames_per_move):
            # A damped spring-like force attracts the token to the destination.
            dx = target_xy[0] - body.position.x
            dy = target_xy[1] - body.position.y
            body.apply_force_at_world_point((dx * 55, dy * 55), body.position)
            space.step(1 / 20)
            frames.append(make_chess_frame(board, {
                "from_sq": move.from_square, "piece": moving_piece,
                "xy": (body.position.x, body.position.y)
            }))

        board.push(move)  # Chess rules take precedence over the visual simulation.
        frames.extend([make_chess_frame(board)] * 3)

    if frames:
        buffer = io.BytesIO()
        frames[0].save(buffer, format="GIF", save_all=True,
                       append_images=frames[1:], duration=80, loop=0)
        display(IPyImage(data=buffer.getvalue()))
        print(f"Animated {min(max_moves, len(episode['moves']))} legal moves.")


show_physics_gif({"moves": [r["move"] for r in demo["trace"]]}, max_moves=4)

## 10. Presentation summary: what strategy did we use?

**Selected learning algorithm:** First-Visit Monte Carlo **Control**.

**Selected policy:** ε-greedy for **both White and Black**; epsilon decreases through training. At every episode:

1. Generate a complete (possibly horizon-adjudicated) chess episode using the current ε-greedy policies.
2. Compute the terminal rewards (+1/0/−1) separately for White and Black.
3. For each player, process each **state–action pair only at its first occurrence in that episode**.
4. Average observed returns to estimate `Q(s, a)` for the agent's own decisions.
5. Improve the next episode's action selection by choosing higher-Q actions more often.

\[
Q(s,a)\leftarrow\frac{\text{sum of observed first-visit returns for }(s,a)}{\text{number of first visits to }(s,a)}
\]

**Difference from the earlier notebook:** The previous version used **First-Visit MC Prediction** to estimate `V(s)` while still choosing random moves. This version uses **First-Visit MC Control** to estimate `Q(s,a)` **and improve the policy**. **Difference from Q-Learning:** MC waits for the episode to finish; Q-Learning uses bootstrapped temporal-difference updates after transitions.

**Caution:** Since states and moves are grouped and training is limited, the results are illustrative, not optimal chess play. To create a strong chess agent, substantial representation learning or search would be needed.